# PHANTOM From Scratch - Train Your Own LLM

Train a cybersecurity LLM from **scratch** - no pretrained weights!

**Runtime: GPU (T4 required)**
---


In [ ]:
# @title ## Setup
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!git clone https://github.com/Njap-png/Cogitrongit.git
%cd Cogitrongit

In [ ]:
# @title Install Dependencies
# @markdown 
!pip install -q torch transformers accelerate datasets peft tokenizers

In [ ]:
# @title Prepare Cybersecurity Training Data
import os
import json

# Load your Q&A data
qa_data = []
with open('data/phantom_train.jsonl', 'r') as f:
    for line in f:
        qa_data.append(json.loads(line))

# Create cybersecurity text corpus
cyber_texts = []

# Add Q&A pairs as training text
for item in qa_data:
    text = f"Instruction: {item['instruction']}\n"
    if item.get('input'):
        text += f"Input: {item['input']}\n"
    text += f"Output: {item['output']}"
    cyber_texts.append(text)

# Add more cybersecurity knowledge (expand this list!)
cyber_knowledge = """
Cybersecurity is the practice of protecting systems, networks, and programs from digital attacks. 
Common threats include malware, phishing, ransomware, and SQL injection. 
Defense in depth uses multiple layers of security. 
The CIA triad is confidentiality, integrity, and availability. 
OWASP Top 10 lists the most critical web application security risks. 
Penetration testing simulates attacks to find vulnerabilities. 
Encryption protects data at rest and in transit. 
Multi-factor authentication adds security layers. 
Zero trust assumes no implicit trust. 
Network segmentation limits attack spread. 
Security auditing finds weaknesses. 
Incident response handles breaches. 
Vulnerability management finds and fixes flaws. 
Security headers protect web apps. 
Input validation prevents injection. 
Least privilege limits access. 
"""

cyber_texts.append(cyber_knowledge)

# Combine all text
full_corpus = "\n\n".join(cyber_texts)
print(f"Total corpus size: {len(full_corpus)} chars, {len(cyber_texts)} docs")

with open('data/cyber_corpus.txt', 'w') as f:
    f.write(full_corpus)
print("Corpus saved to data/cyber_corpus.txt")

In [ ]:
# @title Train PHANTOM From Scratch
import os
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

MODEL_NAME = "gpt2"  # Smallest GPT-2 - we train FROM SCRATCH
OUTPUT_DIR = "/content/drive/MyDrive/phantom_from_scratch"

print("Loading GPT-2 base model (will train from scratch)...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
print(f"Model parameters: {model.num_parameters():,}")

# Load corpus
dataset = load_dataset("text", data_files="data/cyber_corpus.txt")["train"]

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
dataset = dataset.train_test_split(test_size=0.1)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT is not masked LM
)

# Training from scratch - more epochs for learning
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,  # More epochs to learn from scratch
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    warmup_steps=50,
    report_to="none",
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

print("Training PHANTOM from scratch...")
trainer.train()

# Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# @title Test PHANTOM
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

OUTPUT_DIR = "/content/drive/MyDrive/phantom_from_scratch"

model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR)
tokenizer = GPT2Tokenizer.from_pretrained(OUTPUT_DIR)

def generate_phantom(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0])

# Test
print("Testing PHANTOM:\n")
print("=" * 50)
result = generate_phantom("What is SQL injection?")
print(result)
print("=" * 50)

---
## Usage After Training

1. Download model from Drive
2. Run locally: `python phantom.py`
3. Or create Ollama model

## Expand Training Data

Add more cybersecurity content to `data/cyber_corpus.txt`:
- CVE reports
- Hacktivity writeups
- Security blog posts
- CTF solutions
